# Königsberg, 1736: The Birth of Graph Theory

In 1736, Leonhard Euler answered a question about a Prussian city and, in the process, founded two branches of mathematics: graph theory and topology. The question: *can you walk through Königsberg crossing each of its seven bridges exactly once, and return to where you started?*

This notebook builds Königsberg as a graph, verifies Euler's theorem, and lets you poke at the bridges to see when the walk becomes possible.

**Accompanies the lesson:** [`/lessons/graph-theory/koenigsberg`](https://lutchet.netlify.app/lessons/graph-theory/koenigsberg)

In [ ]:
import sys
sys.path.insert(0, '..')

from src.graph_theory.core import Graph, has_eulerian_path, has_eulerian_circuit
from src.graph_theory.koenigsberg import koenigsberg_graph, NORTH, SOUTH, KNEIPHOF, LOMSE

## 1. The city as Euler saw it

Four land masses, seven bridges. Euler's move: forget the geography. Keep only which land masses are connected, and how many bridges connect each pair.

In [ ]:
g = koenigsberg_graph()

print('Land masses:', g.nodes)
print('Bridges:   ', len(g.edges))
print()
print('Which bridge connects to which:')
for u, v in sorted(g.edges):
    print(f'  {u} - {v}')

## 2. Count the degrees

The **degree** of a land mass is the number of bridges meeting it. This is the only thing Euler's proof cares about.

In [ ]:
labels = {
    NORTH: 'North bank',
    SOUTH: 'South bank',
    KNEIPHOF: 'Kneiphof',
    LOMSE: 'Lomse',
}

for node, name in labels.items():
    d = g.degree(node)
    parity = 'odd' if d % 2 else 'even'
    print(f'  {name:12s}  degree = {d}  ({parity})')

print()
print('Sum of degrees =', sum(g.degree(n) for n in g.nodes), '= 2 × number of bridges')

## 3. Euler's theorem

Euler's rule: a connected graph admits a walk crossing every edge exactly once (an **Eulerian path**) if and only if it has zero or two nodes of odd degree. If the walk must also return to its starting node (an **Eulerian circuit**), every node must have even degree.

Königsberg has four nodes of odd degree. Both conditions fail.

In [ ]:
odd = g.odd_degree_nodes()

print(f'Odd-degree nodes: {odd}  ({len(odd)} total; Euler needs 0 or 2)')
print(f'Eulerian path exists?   {has_eulerian_path(g)}')
print(f'Eulerian circuit exists? {has_eulerian_circuit(g)}')

## 4. What would have to change?

Remove one bridge between Kneiphof and the South bank. That drops the Kneiphof degree from 5 to 4 (even) and the South bank degree from 3 to 2 (even). Now the North bank and Lomse are the only odd-degree nodes, so an Eulerian path becomes possible, starting at one and ending at the other.

In [ ]:
g2 = koenigsberg_graph()
g2.remove_edge(SOUTH, KNEIPHOF)   # demolish Köttelbrücke

for node, name in labels.items():
    print(f'  {name:12s}  degree = {g2.degree(node)}')

print()
print('Eulerian path exists?   ', has_eulerian_path(g2))
print('Eulerian circuit exists?', has_eulerian_circuit(g2))

## 5. Can we make the circuit possible?

For an Eulerian circuit, *every* land mass must have even degree. Königsberg's original degrees are 3, 3, 5, 3. We need to remove (or add) bridges until all four are even.

In [ ]:
g3 = koenigsberg_graph()

# Remove one Lomse bridge (drops Lomse 3→2, partner 3→2 or 5→4).
# Do this twice and we get all-even degrees.
g3.remove_edge(NORTH, LOMSE)     # demolish Hohe Brücke
g3.remove_edge(SOUTH, LOMSE)     # demolish Holzbrücke

print('Degrees after removing the two Lomse bridges not to Kneiphof:')
for node, name in labels.items():
    print(f'  {name:12s}  degree = {g3.degree(node)}')

print()
print('Eulerian path exists?   ', has_eulerian_path(g3))
print('Eulerian circuit exists?', has_eulerian_circuit(g3))

## 6. A clean Eulerian circuit from scratch

A square (four nodes, four edges forming a loop) is the simplest non-trivial graph with an Eulerian circuit. Every node has degree 2.

In [ ]:
square = Graph()
square.add_edge('A', 'B')
square.add_edge('B', 'C')
square.add_edge('C', 'D')
square.add_edge('D', 'A')

print('Degrees:', {n: square.degree(n) for n in square.nodes})
print('Eulerian circuit exists?', has_eulerian_circuit(square))

## 7. Your turn

A few things to try:

- Add a *fifth* land mass connected to Königsberg by a single new bridge. What happens to the parity count?
- Find the *smallest* number of bridges you can remove from the original Königsberg graph to make an Eulerian path possible. (Hint: you need to drop odd-degree nodes from 4 down to 2 or 0.)
- For each bridge in Königsberg, check whether removing it alone makes the walk possible. Is there a pattern?

The next story in this topic takes us from *whether* a walk exists to *which walk is shortest*: Dijkstra's algorithm, and the networks that made it matter.